### Setup

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras

2024-09-28 20:01:57.810289: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


### Download the data

In [ ]:
# !!curl -o http://www.manythings.org/anki/fra-eng.zip
# !!unzip fra-eng.zip
# ref: https://www.kaggle.com/code/akshat0007/machine-translation-english-to-french-rnn-lstm/notebook

### Configuration

In [48]:
batch_size = 64 # Batch size for training
epochs = 5 #100 # Number of epochs to train for.
laten_dim = 256 # Laten dimensionality of the encoding space.
num_sample = 10000 # Number of samples to train on.
data_path = "../dataset/fra-eng/fra.txt"

### Prepare the data

In [3]:
# Vectorize the data
input_texts = []
target_texts = []
input_characters = set()
target_characters = set()
with open(data_path, "r", encoding="utf-8") as f:
    lines = f.read().split("\n")
for line in lines[: min(num_sample, len(lines) - 1)]:
    input_text, target_text, _ = line.split("\t")
    # We use "tab as the start sequence" character
    # for the targets, and "\n" as "end sequence" character.
    target_text = "\t" + target_text + "\n"
    input_texts.append(input_text)
    target_texts.append(target_text)
    for char in input_text:
        if char not in input_characters:
            input_characters.add(char)
    for char in target_text:
        if char not in target_characters:
            target_characters.add(char)
            


In [4]:
input_characters = sorted(list(input_characters))
target_characters = sorted(list(target_characters))
num_encoder_tokens = len(input_characters)
num_decoder_tokens = len(target_characters)
max_encoder_seq_length = max([len(txt) for txt in input_texts])
max_decoder_seq_length = max([len(txt) for txt in target_texts])

In [5]:
print("Number of samples: ", len(input_texts))
print("Number of unique input tokens:", num_encoder_tokens)
print("Number of unique output tokens:", num_decoder_tokens)
print("Max sequence length for inputs:", max_encoder_seq_length)
print("Max sequence length for outputs:", max_decoder_seq_length)

Number of samples:  10000
Number of unique input tokens: 70
Number of unique output tokens: 91
Max sequence length for inputs: 14
Max sequence length for outputs: 59


### Input Tokens

In [6]:
input_token_index = dict([char, i] for i, char in enumerate(input_characters)) 
target_token_index = dict([(char, i) for i, char in enumerate(target_characters)])

In [26]:
# input_token_index, target_token_index

### Encoder and Decoder

In [12]:
# refer to blog for more on the parameters: https://blog.keras.io/a-ten-minute-introduction-to-sequence-to-sequence-learning-in-keras.html

encoder_input_data = np.zeros(
    (len(input_texts), max_encoder_seq_length, num_encoder_tokens), dtype="float32"
)
decoder_input_data = np.zeros(
    (len(input_texts), max_decoder_seq_length, num_decoder_tokens), dtype="float32"
)
decoder_target_data = np.zeros(
    (len(input_texts), max_decoder_seq_length, num_decoder_tokens), dtype="float32"
)

In [38]:
# log
print("encoder input data:", encoder_input_data.shape)
print("decoder input data:", decoder_input_data.shape)
print("decoder target data:", decoder_target_data.shape)

encoder input data: (10000, 14, 70)
decoder input data: (10000, 59, 91)
decoder target data: (10000, 59, 91)


### One-Hot Representation

In [8]:
# You can simply use the keras One-hot encoder to do same thing: https://machinelearningmatery.com/how-to..

for i, (input_text, target_text) in enumerate(zip(input_texts, target_texts)):
    for t, char in enumerate(input_text):
        encoder_input_data[i, t, input_token_index[char]] = 1.0
    encoder_input_data[i, t+1:, input_token_index[" "]] = 1.0
    for t, char in enumerate(target_text):
        # decoder_target_data is ahead of decoder_input_data by one time step
        decoder_input_data[i, t, target_token_index[char]] = 1.0
        if t > 0:
            # decoder_target_data will be a head by one time step
            # and will not include the start character.
            decoder_target_data[i, t - 1, target_token_index[char]] = 1.0
    decoder_input_data[i, t + 1 :, target_token_index[" "]] = 1.0
    decoder_target_data[i, t:, target_token_index[" "]] = 1.0

In [85]:
print("encoder input data: ", encoder_input_data.shape)
print(decoder_input_data.shape)
print(decoder_target_data.shape)
print(num_decoder_tokens)
print(laten_dim)
print(num_decoder_tokens)

encoder input data:  (10000, 14, 70)
(10000, 59, 91)
(10000, 59, 91)
91
256
91


### Build the model

In [87]:
# all shape should be equal for model learning
num_decoder_tokens = num_encoder_tokens

print("shape: ", num_decoder_tokens, num_encoder_tokens)


# Define an input sequence and process it.
encoder_inputs = keras.Input(shape=(None, num_encoder_tokens)) #encoder input shape
encoder = keras.layers.LSTM(laten_dim, return_state=True) #specify the laten_dim(your timestamp)
encoder_outputs, state_h, state_c = encoder(encoder_inputs) # encoder returns encoder output, hidden state and cell state

# We discard `encoder_outputs` and only keep the states.
encoder_states = [state_h, state_c] #only keep the hidden and cell state

# Set up the decoder, using `encoder_states` as initial state.
decoder_inputs = keras.Input(shape=(None, num_decoder_tokens))

# We set up our decoder to return full output sequences,
# and to return internal states as well. We don't use the 
# return states in the training model, but we will use them in inference.
decoder_lstm = keras.layers.LSTM(laten_dim, return_sequences=True, return_state=True)
decoder_outputs,_,_ = decoder_lstm(decoder_inputs, initial_state=encoder_states)

decoder_dense = keras.layers.Dense(num_decoder_tokens, activation="softmax") #adding dense layer
decoder_outputs = decoder_dense(decoder_outputs) #getting all the output

print("encoder input: ", encoder_inputs)
print("decoder inputs: ", decoder_inputs)
print('decoder output:', decoder_outputs)

# Define the model that will turn
# `encoder_input_data` & `decoder_input_data` into `decoder_target_data`
model = keras.Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.summary()


shape:  70 70
encoder input:  <KerasTensor shape=(None, None, 70), dtype=float32, sparse=False, name=keras_tensor_306>
decoder inputs:  <KerasTensor shape=(None, None, 70), dtype=float32, sparse=False, name=keras_tensor_310>
decoder output: <KerasTensor shape=(None, None, 70), dtype=float32, sparse=False, name=keras_tensor_314>


Model: "functional_17"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_16      │ (None, None, 70)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_17      │ (None, None, 70)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_12 (LSTM)      │ [(None, 256),     │    334,848 │ input_layer_16[0… │
│                     │ (None, 256),      │            │                   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_13 (LSTM)      │ [(None, None,     │    334,848 │ input_layer_17[0… │
│                     │ 256), (None,      │            │ lstm_12[0][1],    │
│                     │ 256), (None,      │            │ lstm_12[0][2]     │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, None, 70)  │     17,990 │ lstm_13[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 687,686 (2.62 MB)

 Trainable params: 687,686 (2.62 MB)

 Non-trainable params: 0 (0.00 B)

In [88]:
print("encoder input: ", encoder_inputs)
print("decoder input: ", decoder_inputs),
print("decoder output: ", decoder_outputs)
print(encoder_input_data.shape)
print(decoder_input_data.shape)
print(decoder_target_data.shape)

encoder input:  <KerasTensor shape=(None, None, 70), dtype=float32, sparse=False, name=keras_tensor_306>
decoder input:  <KerasTensor shape=(None, None, 70), dtype=float32, sparse=False, name=keras_tensor_310>
decoder output:  <KerasTensor shape=(None, None, 70), dtype=float32, sparse=False, name=keras_tensor_314>
(10000, 14, 70)
(10000, 59, 91)
(10000, 59, 91)


In [89]:
from sklearn.decomposition import PCA

# Apply PCA to reduce dimensions from 91 to 70 for decoder input data
decoder_input_data_flat = decoder_input_data.reshape(-1, 91)  # Flatten
pca = PCA(n_components=70)
decoder_input_data_reduced = pca.fit_transform(decoder_input_data_flat)

# Reshape back to original format
decoder_input_data_reduced = decoder_input_data_reduced.reshape(10000, 59, 70)

# Repeat for decoder target data
decoder_target_data_flat = decoder_target_data.reshape(-1, 91)
decoder_target_data_reduced = pca.transform(decoder_target_data_flat)
decoder_target_data_reduced = decoder_target_data_reduced.reshape(10000, 59, 70)

print("Reduced decoder input shape:", decoder_input_data_reduced.shape)
print("Reduced decoder target shape:", decoder_target_data_reduced.shape)


/Users/rattanak/Documents/Sample_Projects/python_for_cs/.venv/lib/python3.12/site-packages/sklearn/decomposition/_pca.py:653: RuntimeWarning: invalid value encountered in divide
  explained_variance_ratio_ = explained_variance_ / total_var


Reduced decoder input shape: (10000, 59, 70)
Reduced decoder target shape: (10000, 59, 70)


### Train the model

In [90]:
from tensorflow.keras.optimizers import Adam
optimizer = Adam(learning_rate=0.001)  # Adjust the learning rate
model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])


# model.compile(
#     optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"]
# )

# model.fit(
#     [encoder_input_data, decoder_input_data],
#     decoder_target_data,
#     batch_size=batch_size,
#     epochs=epochs,
#     validation_split=0.2
# )

# reduce size data input
model.fit(
    [encoder_input_data, decoder_input_data_reduced],
    decoder_target_data_reduced,
    batch_size=batch_size,
    epochs=epochs,
    validation_split=0.2
)

# Save model
model.save("s2s_translate_fra-en.keras")

Epoch 1/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 29s 212ms/step - accuracy: 0.0430 - loss: 0.0000e+00 - val_accuracy: 0.0000e+00 - val_loss: 0.0000e+00
Epoch 2/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 36s 286ms/step - accuracy: 0.0000e+00 - loss: 0.0000e+00 - val_accuracy: 0.0000e+00 - val_loss: 0.0000e+00
Epoch 3/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 42s 335ms/step - accuracy: 0.0000e+00 - loss: 0.0000e+00 - val_accuracy: 0.0000e+00 - val_loss: 0.0000e+00
Epoch 4/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 37s 296ms/step - accuracy: 0.0000e+00 - loss: 0.0000e+00 - val_accuracy: 0.0000e+00 - val_loss: 0.0000e+00
Epoch 5/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 33s 261ms/step - accuracy: 0.0000e+00 - loss: 0.0000e+00 - val_accuracy: 0.0000e+00 - val_loss: 0.0000e+00


### Run inference (sampling)
1. encode input and retrieve initial decoder state
2. run one step of  decoder with this initial state and a "start of sequence" token as target. Output will be the next target token.
3. Repeat with the current target token and current states

In [91]:
# Load the pre-trained model
model = keras.models.load_model("s2s_translate_fra-en.keras")
# Extract the encoder inputs (typically the first input layer)
encoder_inputs = model.input[0]  # This is the encoder input layer (input_1)
# Extract the encoder LSTM layers' output and states
encoder_outputs, state_h_enc, state_c_enc = model.layers[2].output  # lstm_1 (ensure it's the correct LSTM layer)
# Combine the LSTM states for later use
encoder_states = [state_h_enc, state_c_enc]
# Build the encoder model from the inputs to the encoder states
encoder_model = keras.Model(encoder_inputs, encoder_states)
# Print to verify or debug
print(encoder_model.summary())

#input 2
decoder_inputs = model.input[1] # input_2
decoder_state_input_h = keras.Input(shape=(laten_dim,))
decoder_state_input_c = keras.Input(shape=(laten_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]
decoder_lstm = model.layers[3]
decoder_outputs, state_h_dec, state_c_dec = decoder_lstm(
    decoder_inputs, initial_state=decoder_states_inputs
)
decoder_states = [state_h_dec, state_c_dec]
decoder_dense = model.layers[4]
decoder_outputs = decoder_dense(decoder_outputs)
decoder_model = keras.Model(
    [decoder_inputs] + decoder_states_inputs, [decoder_outputs] + decoder_states
)

Model: "functional_18"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_16 (InputLayer)     │ (None, None, 70)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_12 (LSTM)                  │ [(None, 256), (None,   │       334,848 │
│                                 │ 256), (None, 256)]     │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 334,848 (1.28 MB)

 Trainable params: 334,848 (1.28 MB)

 Non-trainable params: 0 (0.00 B)

None


In [92]:
# Reverse-lookup token index to decode sequence back to
# something readable
reverse_input_char_index = dict((i, char) for char, i in input_token_index.items())
reverse_target_char_index = dict((i, char)  for char, i in target_token_index.items())
reverse_input_char_index

def decode_sequence(input_seq):
    # Encode the input as state vectors.
    states_value = encoder_model.predict(input_seq)

    # Generate empty target sequence of length 1.
    target_seq = np.zeros((1, 1, num_decoder_tokens))
    # Populate the first character of target sequence with the start character
    target_seq[0, 0, target_token_index["\t"]] = 1.0

    # Sampling loop for a batch of sequences
    # (to simplify, here we assume a batch of size 1).
    stop_condition = False
    decoded_sentence = ""
    while not stop_condition:
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value)

        # Sample a token
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_char = reverse_target_char_index[sampled_token_index]
        decoded_sentence += sampled_char

        # Exit condition: either hit max length
        # or find stop character.
        if sampled_char == "\n" or len(decoded_sentence) > max_decoder_seq_length:
            stop_condition = True

        # Update the target sequence (of length 1):
        target_seq = np.zeros((1, 1, num_decoder_tokens))
        target_seq[0, 0, sampled_token_index] = 1.0

        # update states
        states_value = [h, c]

    return decoded_sentence

In [93]:
# you can now generate decoded sentences as such:

for seq_index in range(1):
    # print("seq_index", seq_index)
    # Take one sequence (part of the training set)
    # for trying out decoding.
    input_seq = encoder_input_data[seq_index : seq_index + 1]
    # print("input seq: ", input_seq)
    decoded_sentence = decode_sequence(input_seq)
    print("-")
    print("Input Sentence: ", input_texts[seq_index])
    print("Decoded sentence: ", decoded_sentence)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 154ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 202ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
1/1 ━━━━━━

In [74]:
encoder_input_data

array([[[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]],

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]],

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]],

       ...,

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0.